# Notebook 05: Parameter-Efficient Fine-Tuning (PEFT) with LoRA

In this notebook, we use **LoRA (Low-Rank Adaptation)** to fine-tune the Moirai encoder.
Instead of updating millions of parameters, LoRA freezes the foundation model and injects tiny trainable rank-decomposition matrices into the linear layers.

uv pip install peft

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import copy
from IPython.display import display


from tslearn.datasets import UCR_UEA_datasets
from sklearn.preprocessing import LabelEncoder
from uni2ts.model.moirai import MoiraiModule
from encoder import MoiraiEncoder

from heads import *
from models import *
from utils import *

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES = [16] #[8, 16, 32, 64]

lr_grid=[5e-5]
wd_grid=[0.05]

BATCH_SIZE = 32 

In [ ]:
tr_loader, va_loader, te_loader, num_classes = get_lsst_dataloaders(BATCH_SIZE, DEVICE)

## 2. LoRA Baseline: Mean Pooling on Full Sequence (Context + Mask)

In [ ]:
df_mean_pool = pd.DataFrame(index=["Mean Pooling (LoRA)"], columns=PATCH_SIZES)
df_mean_pool.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    _, test_acc = universal_grid_search(
        model_class=LoraHeadWrapper,
        model_kwargs={
            "head_class": MeanPoolingClassifier,
            "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
            "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, 
            "remove_last_patch": False, "lora_r": 8
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid= wd_grid, lr_grid = lr_grid
    )
    df_mean_pool.loc["Mean Pooling (LoRA)", patch] = test_acc

trainable params: 350,208 || all params: 14,177,736 || trainable%: 2.4701
 Early stopping : 48
trainable params: 350,208 || all params: 14,177,736 || trainable%: 2.4701
 Early stopping : 44


Patch Size,8,16
Mean Pooling (LoRA),0.6464,0.6464


In [ ]:
display(df_mean_pool.astype(float).round(4))

## 3. LoRA Rank Ablation Study
Testing the impact of the bottleneck rank $r$ on the MHA performance. A higher rank means more trainable parameters.

In [ ]:
PATCH = 8
RANKS_TO_TEST = [4, 8, 16, 32, 64] 

df_rank_ablation = pd.DataFrame(index=[f"MHA ({MODE})"], columns=RANKS_TO_TEST)
df_rank_ablation.index.name = "Model"
df_rank_ablation.columns.name = f"LoRA Rank 'r' (Patch {PATCH})"

for r in RANKS_TO_TEST:
    _, test_acc = universal_grid_search(
        model_class=LoraHeadWrapper,
        model_kwargs={
            "head_class": MeanPoolingClassifier,
            "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
            "patch_size": PATCH, "num_vars": NUM_VARS, "size": SIZE, 
            "remove_last_patch": False, "lora_r": r
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        lr_grid=lr_grid, wd_grid = wd_grid
    )
    df_rank_ablation.loc[f"MHA ({MODE})", r] = test_acc


--- Testing LoRA Rank: 4 ---
trainable params: 175,104 || all params: 14,002,632 || trainable%: 1.2505


 Early stopping : 49
0.6411192214111923

--- Testing LoRA Rank: 8 ---
trainable params: 350,208 || all params: 14,177,736 || trainable%: 2.4701
 Early stopping : 45
0.6374695863746959

--- Testing LoRA Rank: 16 ---
trainable params: 700,416 || all params: 14,527,944 || trainable%: 4.8212
 Early stopping : 40
0.648418491484185

--- Testing LoRA Rank: 32 ---
trainable params: 1,400,832 || all params: 15,228,360 || trainable%: 9.1988
 Early stopping : 40
0.6557177615571776

--- Testing LoRA Rank: 64 ---
trainable params: 2,801,664 || all params: 16,629,192 || trainable%: 16.8479
 Early stopping : 37
0.64882400648824


LoRA Rank 'r' (Patch 8),4,8,16,32,64
Model,,,,,
MHA (independent_context),0.6411,0.6375,0.6484,0.6557,0.6488


In [ ]:
display(df_rank_ablation.astype(float).round(4))

## 4. Advanced LoRA: Multi-Head Attention (Context + Mask)

In [ ]:
PATCH_SIZES_TO_TEST = [8, 16] 
MODES = ["shared_context", "independent_context"]
NUM_HEADS = 16

model_names_single = [f"MHA (Heads={NUM_HEADS} | LoRA r=8)"]
index_single = pd.MultiIndex.from_product([model_names_single, MODES], names=["Model", "Mode"])
df_adv_single = pd.DataFrame(index=index_single, columns=PATCH_SIZES_TO_TEST)
df_adv_single.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        _, acc_mha = universal_grid_search(
            model_class=LoraHeadWrapper,
            model_kwargs={
                "head_class": SingleScaleMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, 
                "remove_last_patch": False, "lora_r": 8
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_adv_single.loc[(model_names_single[0], mode), patch] = acc_mha

trainable params: 350,208 || all params: 14,177,736 || trainable%: 2.4701
 Early stopping : 39
trainable params: 350,208 || all params: 14,177,736 || trainable%: 2.4701
 Early stopping : 38
trainable params: 350,208 || all params: 14,177,736 || trainable%: 2.4701
 Early stopping : 40
trainable params: 350,208 || all params: 14,177,736 || trainable%: 2.4701
 Early stopping : 38


Patch Size                                         8       16
Model                     Mode                               
MHA (Heads=16 | LoRA r=8) shared_context       0.6204  0.6172
                          independent_context  0.6123  0.6139

In [ ]:
display(df_adv_single.astype(float).round(4))